## Third Question

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

##### First Task
###### 1. Create a Data Frame with custom schema creation by using Struct Type and Struct Field 

In [0]:
login_data = [
    (1, 101, 'login', '2023-09-05 08:30:00'), 
    (2, 102, 'click', '2023-09-06 12:45:00'), 
    (3, 101, 'click', '2023-09-07 14:15:00'), 
    (4, 103, 'login', '2023-09-08 09:00:00'), 
    (5, 102, 'logout', '2023-09-09 17:30:00'), 
    (6, 101, 'click', '2023-09-10 11:20:00'), 
    (7, 103, 'click', '2023-09-11 10:15:00'), 
    (8, 102, 'click', '2023-09-12 13:10:00') 
]

schema = StructType([
    StructField("log id", IntegerType(), True),
    StructField("user$id", IntegerType(), True),
    StructField("action", StringType(), True),
    StructField("timestamp", StringType(), True)
])

df = spark.createDataFrame(login_data, schema)
display(df)

##### Second Task
###### 2. Column names should be log_id, user_id, user_activity, time_stamp using dynamic function

In [0]:
df = df.withColumnsRenamed({"log id": "log_id", "user$id": "user_id", "action": "user_activiry", "timestamp": "time_stamp"})
display(df)

##### Third Task
###### 3. Write a query to calculate the number of actions performed by each user in the last 7 days

In [0]:
df = df.withColumn(
    "time_stamp",
    to_timestamp("time_stamp", "yyyy-MM-dd HH:mm:ss")
)

latest_date = df.select(max("time_stamp")).first()[0]

last_7_days_df = df.filter(
    col("time_stamp") >= date_sub(lit(latest_date), 7)
)

last_login_df = (
    last_7_days_df
    .groupBy("user_id")
    .agg(count("*").alias("total_actions"))
)
display(last_login_df)

##### Fourt Task
###### 4. Convert the time stamp column to the login_date column with YYYY-MM-DD format with date type as its data type 

In [0]:
df = df.withColumn(
    "login_date",
    to_date("time_stamp")
)
display(df)

##### Fifth Task
###### 5. Write the data frame as a CSV file with different write options except (merge condition)

In [0]:
df.write.mode("overwrite")\
    .option("header", "true")\
    .option("delimiter", "|")\
    .option("nullValue", "NULL")\
    .option("encoding", "UTF-8")\
    .option("quote", "\"")\
    .option("escape", "\"")\
    .csv("/Volumes/workspace/default/my_volume/login_data")
    # .csv("/Workspace/Users/jaygediya1802@gmail.com/pysparkAssignment/login_data")

##### Sixed Task
###### 6. Write it as a managed table with the Database name as user and table name as login_details with overwrite mode. 

In [0]:
df.write.mode("overwrite")\
    .saveAsTable("login_data_table")